In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Lector Kafka Avro a Delta
# MAGIC
# MAGIC Este notebook consume eventos Avro del topic `banco.pagos`, consulta el
# MAGIC esquema en Schema Registry y guarda los eventos decodificados en Delta.

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.avro.functions import from_avro


# ============================================================
# CONFIGURACIÓN: modificar estos valores antes de ejecutar
# ============================================================

# Conexión al servicio Apache Kafka de Aiven.
KAFKA_BOOTSTRAP_SERVERS = "kafka-utec-pe-01-utec-data.b.aivencloud.com:28780"
KAFKA_TOPIC = "banco.pagos"
KAFKA_STARTING_OFFSETS = "earliest"  # "earliest" lee todo; "latest" solo lo nuevo.
KAFKA_MAX_OFFSETS_PER_TRIGGER = 1000  # Límite de eventos por micro-lote.
KAFKA_FAIL_ON_DATA_LOSS = "false"  # Evita detener el stream si Kafka elimina offsets antiguos.

# Autenticación SASL. En Aiven normalmente se usa SCRAM-SHA-256.
KAFKA_SECURITY_PROTOCOL = "SASL_SSL"
KAFKA_SASL_MECHANISMS = "SCRAM-SHA-256"
KAFKA_SASL_USERNAME = "avnadmin"
KAFKA_SASL_PASSWORD = dbutils.secrets.get(catalog="dev_bronze", schema="kafka_streams", key="KAFKA_SASL_PASSWORD")
KAFKA_JAAS_LOGIN_MODULE = "kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule" # Para databricks siempre poner le prefijo "kafkashaded"

# Certificado CA de Aiven en formato PEM.
# Pegar el contenido completo, incluyendo BEGIN CERTIFICATE y END CERTIFICATE.
KAFKA_CA_CERTIFICATE_PEM = dbutils.secrets.get(catalog="dev_bronze", schema="kafka_streams", key="KAFKA_CA_CERTIFICATE_PEM")

# Conexión a Schema Registry. La URL NO debe incluir usuario ni contraseña.
SCHEMA_REGISTRY_URL = "https://kafka-utec-pe-01-utec-data.b.aivencloud.com:28772"
SCHEMA_REGISTRY_USERNAME = "avnadmin"
SCHEMA_REGISTRY_PASSWORD = dbutils.secrets.get(catalog="dev_bronze", schema="kafka_streams", key="SCHEMA_REGISTRY_PASSWORD")

# AvroSerializer usa por defecto el subject <topic>-value.
SCHEMA_SUBJECT = f"{KAFKA_TOPIC}-value"

# Destino Delta y estado del streaming. El checkpoint debe ser único y persistente.
CATALOG = "dev_bronze"
SCHEMA = "kafka_streams"
TABLE_NAME = KAFKA_TOPIC.replace(".","_")
DELTA_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
CHECKPOINT_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA}/vol_streams/checkpoints/{TABLE_NAME}"
TRIGGER_INTERVAL = "10 seconds"
QUERY_NAME = "bronze_pagos_kafka"


# ============================================================
# LECTURA DE KAFKA
# ============================================================

# Kafka usa este texto para autenticar el usuario mediante SASL/SCRAM.
KAFKA_SASL_JAAS_CONFIG = (
    f'{KAFKA_JAAS_LOGIN_MODULE} required '
    f'username="{KAFKA_SASL_USERNAME}" '
    f'password="{KAFKA_SASL_PASSWORD}";'
)

# Spark reenvía las opciones que empiezan con kafka. al cliente Kafka de Java.
# El certificado se entrega como valor PEM, sin requerir una ruta de archivo.
kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", KAFKA_STARTING_OFFSETS)
    .option("maxOffsetsPerTrigger", KAFKA_MAX_OFFSETS_PER_TRIGGER)
    .option("failOnDataLoss", KAFKA_FAIL_ON_DATA_LOSS)
    .option("kafka.security.protocol", KAFKA_SECURITY_PROTOCOL)
    .option("kafka.sasl.mechanism", KAFKA_SASL_MECHANISMS)
    .option("kafka.sasl.jaas.config", KAFKA_SASL_JAAS_CONFIG)
    .option("kafka.ssl.truststore.type", "PEM")
    .option("kafka.ssl.truststore.certificates", KAFKA_CA_CERTIFICATE_PEM)
    .load()
)


# ============================================================
# DECODIFICACIÓN AVRO
# ============================================================

# Credenciales para que from_avro consulte el subject en Schema Registry.
schema_registry_options = {
    "confluent.schema.registry.basic.auth.credentials.source": "USER_INFO",
    "confluent.schema.registry.basic.auth.user.info": (
        f"{SCHEMA_REGISTRY_USERNAME}:{SCHEMA_REGISTRY_PASSWORD}"
    ),
    "mode": "FAILFAST",
    "avroSchemaEvolutionMode": "restart",
}

# Kafka entrega key y value como bytes. from_avro usa Schema Registry para:
# 1. Leer el ID de esquema incluido por AvroSerializer.
# 2. Obtener el esquema del subject banco.pagos-value.
# 3. Convertir los bytes en columnas Spark.
eventos_avro_df = kafka_df.select(
    F.col("topic").alias("kafka_topic"),
    F.col("partition").alias("kafka_partition"),
    F.col("offset").alias("kafka_offset"),
    F.col("timestamp").alias("kafka_timestamp"),
    F.col("key").cast("string").alias("prestamo_key"),
    from_avro(
        data=F.col("value"),
        jsonFormatSchema=None,
        options=schema_registry_options,
        subject=SCHEMA_SUBJECT,
        schemaRegistryAddress=SCHEMA_REGISTRY_URL,
    ).alias("evento"),
)

# evento es una estructura. evento.* la convierte en columnas Delta normales.
eventos_delta_df = (
    eventos_avro_df
    .select("evento.*", "kafka_topic", "kafka_partition", "kafka_offset",
            "kafka_timestamp", "prestamo_key")
    .withColumn("fecha_ingesta", F.current_timestamp())
)


# ============================================================
# ESCRITURA DELTA
# ============================================================

# append conserva todos los eventos. checkpoint guarda los offsets procesados
# para que el stream pueda reiniciarse sin volver a procesar los mismos mensajes.
query = (
    eventos_delta_df.writeStream
    .queryName(QUERY_NAME)
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(processingTime='15 seconds') #.trigger(processingTime='10 seconds') .trigger(availableNow=True)
    .toTable(DELTA_TABLE)
)

print(f"Stream iniciado: {QUERY_NAME}")
print(f"Kafka topic: {KAFKA_TOPIC}")
print(f"Tabla Delta: {DELTA_TABLE}")
print("Para detenerlo en otra celda: query.stop()")
